# Experimentos: PINN/QPINN/HQNN para Black–Scholes

Notebook organizado por seção de experimento, sem CQNN, com:

- seleção de arquiteturas já rodadas (`hidden`, `blocks`, `n_qubits`, `n_layers`, `seed` etc.);
- dados reais de mercado NDX;
- simulação Black–Scholes 1D;
- Greeks sintéticas e de mercado;
- comparação estilo Trahan por erro vs parâmetros;
- teste sintético: erro médio absoluto de preço menor que 1 dólar.


## 0. Setup

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import torch as tc
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import norm

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "experimentos_pinn"
OUT_DIR = PROJECT_ROOT / "figures_experimentos_trahan_bs"
TAB_DIR = PROJECT_ROOT / "tables_experimentos_trahan_bs"
OUT_DIR.mkdir(exist_ok=True)
TAB_DIR.mkdir(exist_ok=True)

# Parâmetros do problema sintético usado no treino
TRAIN_K = 40.0
TRAIN_S_MAX = 160.0
TRAIN_T = 1.0
TRAIN_V_MAX = 120.0
TRAIN_R = 0.05
TRAIN_SIGMA = 0.2
DEVICE = "cpu"

# Famílias usadas no artigo: sem CQNN
FAMILIES = ["CLASSICO", "QNN", "HQNN"]

# Estilo visual parecido com Mathematica
COLORS = {
    "Market": "black",
    "Black-Scholes IV": "#5E81B5",
    "Black-Scholes train σ": "#E19C24",
    "Black-Scholes": "#5E81B5",
    "CLASSICO": "#8FB131",
    "QNN": "#80699B",
    "HQNN": "#C44E52",
}

mpl.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.04,
    "savefig.dpi": 300,
    "figure.dpi": 130,
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "axes.edgecolor": "black",
    "axes.linewidth": 0.9,
    "axes.grid": True,
    "grid.color": "0.84",
    "grid.linewidth": 0.55,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "legend.frameon": False,
})

def savefig(name):
    path = OUT_DIR / name
    plt.tight_layout()
    plt.savefig(path)
    plt.show()
    print(f"[OK] {path}")
    return path


def normalize_family(model_type):
    s = str(model_type)
    if s in ["MLP", "ResNet", "CLASSICO", "CLASSIC", "classico"]:
        return "CLASSICO"
    if s in ["QNN", "QPINN"]:
        return "QNN"
    if s in ["HQNN", "Hybrid", "Hibrido"]:
        return "HQNN"
    return s


def metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y = y_true[mask]
    p = y_pred[mask]
    if len(y) == 0:
        return pd.Series({"N": 0, "RMSE": np.nan, "MAE": np.nan, "Bias": np.nan, "Corr": np.nan})
    err = p - y
    corr = np.nan
    if len(y) > 1 and np.std(y) > 0 and np.std(p) > 0:
        corr = np.corrcoef(y, p)[0, 1]
    return pd.Series({
        "N": len(y),
        "RMSE": np.sqrt(np.mean(err**2)),
        "MAE": np.mean(np.abs(err)),
        "Bias": np.mean(err),
        "Corr": corr,
    })

## 0B. Seleção dos modelos e critério de 1 dólar sintético

Aqui você escolhe quais arquiteturas entram na análise. O notebook continua sem CQNN.  
A seleção é feita **em cima dos modelos já rodados** e do `greeks_summary.csv`; nada é retreinado.


In [ ]:
# ============================================================
# SELEÇÃO DOS MODELOS JÁ RODADOS
# Mude só este bloco quando quiser escolher hidden, blocks, qubits etc.
# ============================================================

# Critério para escolher a melhor run dentro de cada família depois dos filtros.
# Use normalmente RMSE_V_global. Se quiser escolher por erro médio, use MAE_V_global.
SELECTION_SCORE_COL = "RMSE_V_global"

# Critério do teste sintético: erro médio absoluto de preço menor que 1 dólar sintético.
SYNTHETIC_ONE_DOLLAR_THRESHOLD = 1.0

# Se True, quando uma família fica vazia depois dos filtros, usa todas as runs daquela família.
ALLOW_FALLBACK_TO_UNFILTERED_FAMILY = True

# Para escolher runs exatas, preencha os run_id aqui. Se deixar lista vazia, usa os filtros abaixo.
EXACT_RUNS_BY_FAMILY = {
    "CLASSICO": [],
    "QNN": [],
    "HQNN": [],
}

# Filtros por família. Use None para não filtrar uma coluna.
# Exemplo: "hidden": [2, 3, 5] mantém só essas larguras, se a coluna existir.
MODEL_SELECTION = {
    "CLASSICO": {
        "model_type": ["MLP", "ResNet"],
        "hidden": None,       # exemplo: [2, 3, 5, 10]
        "blocks": None,       # exemplo: [1, 2, 3, 4, 5]
        "activation": None,
        "seed": None,
    },
    "QNN": {
        "model_type": ["QNN", "QPINN"],
        "n_qubits": None,     # exemplo: [2, 3, 4, 7, 10]
        "n_layers": None,     # exemplo: [1, 2, 3]
        "entangler": None,
        "seed": None,
    },
    "HQNN": {
        "model_type": ["HQNN"],
        "hidden": None,       # exemplo: [2, 3, 5]
        "blocks": None,       # exemplo: [1, 2, 3, 5]
        "n_qubits": None,     # exemplo: [2, 3, 5]
        "n_layers": None,     # exemplo: [1, 2, 3]
        "entangler": None,
        "seed": None,
    },
}

# Orçamento máximo de parâmetros por família. Use None para não limitar.
MAX_PARAMS_BY_FAMILY = {
    "CLASSICO": None,  # exemplo: 300
    "QNN": None,      # exemplo: 300
    "HQNN": None,     # exemplo: 300
}

print("Seleção configurada. Ajuste este bloco para trocar hidden, blocks, qubits, layers ou seeds.")


## 1. Funções Black–Scholes e predição dos modelos

In [ ]:
def bs_call_greeks_tau(S, tau, K, r, sigma):
    """
    Black-Scholes European call usando tau = tempo até maturidade.
    Theta é a derivada em relação ao tempo calendário t da PDE, isto é, tau = T - t.
    """
    S = np.asarray(S, dtype=float)
    tau = np.asarray(tau, dtype=float)
    K = np.asarray(K, dtype=float)
    r = np.asarray(r, dtype=float)
    sigma = np.asarray(sigma, dtype=float)

    eps = 1e-10
    S_safe = np.maximum(S, eps)
    tau_safe = np.maximum(tau, eps)
    sigma_safe = np.maximum(sigma, eps)

    d1 = (np.log(S_safe / K) + (r + 0.5 * sigma_safe**2) * tau_safe) / (sigma_safe * np.sqrt(tau_safe))
    d2 = d1 - sigma_safe * np.sqrt(tau_safe)

    V = S_safe * norm.cdf(d1) - K * np.exp(-r * tau_safe) * norm.cdf(d2)
    delta = norm.cdf(d1)
    gamma = norm.pdf(d1) / (S_safe * sigma_safe * np.sqrt(tau_safe))
    theta = -(S_safe * norm.pdf(d1) * sigma_safe) / (2 * np.sqrt(tau_safe)) - r * K * np.exp(-r * tau_safe) * norm.cdf(d2)

    V = np.where(tau <= eps, np.maximum(S - K, 0.0), V)
    delta = np.where(tau <= eps, (S > K).astype(float), delta)
    gamma = np.where(tau <= eps, 0.0, gamma)

    return V, delta, gamma, theta


def bs_call_greeks_pde_t(S, t, K=TRAIN_K, r=TRAIN_R, sigma=TRAIN_SIGMA, T=TRAIN_T):
    tau = T - np.asarray(t, dtype=float)
    return bs_call_greeks_tau(S=S, tau=tau, K=K, r=r, sigma=sigma)


def predict_price_greeks_synthetic(model, S, t):
    """
    Predição em escala sintética do treino:
        entrada: S/Smax, t/T
        saída: V/Vmax
    Retorna V, delta, gamma, theta em escala física sintética.
    """
    model.eval()

    S_norm = tc.tensor(np.asarray(S).reshape(-1, 1) / TRAIN_S_MAX, dtype=tc.float32, requires_grad=True)
    t_norm = tc.tensor(np.asarray(t).reshape(-1, 1) / TRAIN_T, dtype=tc.float32, requires_grad=True)
    x = tc.cat([S_norm, t_norm], dim=1)

    V_norm = model(x).reshape(-1, 1)
    ones = tc.ones_like(V_norm)

    dV_dS_norm = tc.autograd.grad(V_norm, S_norm, grad_outputs=ones, create_graph=True, retain_graph=True)[0]
    dV_dt_norm = tc.autograd.grad(V_norm, t_norm, grad_outputs=ones, create_graph=True, retain_graph=True)[0]
    d2V_dS2_norm = tc.autograd.grad(dV_dS_norm, S_norm, grad_outputs=tc.ones_like(dV_dS_norm), create_graph=False, retain_graph=False)[0]

    V = V_norm.detach().cpu().numpy().reshape(-1) * TRAIN_V_MAX
    delta = dV_dS_norm.detach().cpu().numpy().reshape(-1) * (TRAIN_V_MAX / TRAIN_S_MAX)
    gamma = d2V_dS2_norm.detach().cpu().numpy().reshape(-1) * (TRAIN_V_MAX / TRAIN_S_MAX**2)
    theta = dV_dt_norm.detach().cpu().numpy().reshape(-1) * (TRAIN_V_MAX / TRAIN_T)

    return V, delta, gamma, theta


def predict_price_greeks_market_scaled(model, df, scale):
    """
    Mercado reescalado por strike:
        S_scaled = scale * S_real
        V_scaled = scale * V_real
    O modelo prediz V_scaled. Depois voltamos para escala real.
    """
    model.eval()

    S_norm = tc.tensor(df["S_norm"].values.reshape(-1, 1), dtype=tc.float32, requires_grad=True)
    t_norm = tc.tensor(df["t_norm"].values.reshape(-1, 1), dtype=tc.float32, requires_grad=True)
    x = tc.cat([S_norm, t_norm], dim=1)

    V_norm = model(x).reshape(-1, 1)
    ones = tc.ones_like(V_norm)

    dV_dS_norm = tc.autograd.grad(V_norm, S_norm, grad_outputs=ones, create_graph=True, retain_graph=True)[0]
    dV_dt_norm = tc.autograd.grad(V_norm, t_norm, grad_outputs=ones, create_graph=True, retain_graph=True)[0]
    d2V_dS2_norm = tc.autograd.grad(dV_dS_norm, S_norm, grad_outputs=tc.ones_like(dV_dS_norm), create_graph=False, retain_graph=False)[0]

    V_scaled = V_norm.detach().cpu().numpy().reshape(-1) * TRAIN_V_MAX
    delta_scaled = dV_dS_norm.detach().cpu().numpy().reshape(-1) * (TRAIN_V_MAX / TRAIN_S_MAX)
    gamma_scaled = d2V_dS2_norm.detach().cpu().numpy().reshape(-1) * (TRAIN_V_MAX / TRAIN_S_MAX**2)
    theta_scaled = dV_dt_norm.detach().cpu().numpy().reshape(-1) * (TRAIN_V_MAX / TRAIN_T)

    # Volta para escala real
    V_real = V_scaled / scale
    delta_real = delta_scaled
    gamma_real = gamma_scaled * scale
    theta_real = theta_scaled / scale

    return V_real, delta_real, gamma_real, theta_real

## 2. Experimento 0 — dados reais de mercado primeiro

Carrega NDX European Call do projeto, calcula midpoint bid/ask e seleciona o contrato com mais observações.

In [ ]:
from utils.get_data import get_data

MARKET = "NDX"

data_yf, opt = get_data(MARKET)

opt = opt.copy()
opt["date"] = pd.to_datetime(opt["date"])

px = data_yf.reset_index().copy()
px["Date"] = pd.to_datetime(px["Date"])

market_all = opt.merge(px, left_on="date", right_on="Date", how="inner")

market_all = market_all[[
    "date", "Close", "current_time", "strike_price", "best_bid",
    "best_offer", "ticker", "impl_volatility"
]].copy()

market_all = market_all.rename(columns={"Close": "spot_price"})
market_all["strike_price"] = pd.to_numeric(market_all["strike_price"], errors="coerce") / 1000.0
market_all["best_bid"] = pd.to_numeric(market_all["best_bid"], errors="coerce")
market_all["best_offer"] = pd.to_numeric(market_all["best_offer"], errors="coerce")
market_all["market_price"] = (market_all["best_bid"] + market_all["best_offer"]) / 2.0
market_all["tau"] = pd.to_numeric(market_all["current_time"], errors="coerce")

if market_all["tau"].median() > 5:
    market_all["tau"] = market_all["tau"] / 365.0

market_all["impl_volatility"] = pd.to_numeric(market_all["impl_volatility"], errors="coerce")
if market_all["impl_volatility"].median() > 3:
    market_all["impl_volatility"] = market_all["impl_volatility"] / 100.0

market_all["r"] = TRAIN_R
market_all["moneyness"] = market_all["spot_price"] / market_all["strike_price"]

market_all = market_all.dropna(subset=["spot_price", "strike_price", "market_price", "tau", "impl_volatility"]).copy()
market_all = market_all.sort_values(["ticker", "date"]).reset_index(drop=True)

ticker_counts = market_all["ticker"].value_counts().reset_index()
ticker_counts.columns = ["ticker", "N"]
display(ticker_counts.head(15))

ticker_principal = ticker_counts.iloc[0]["ticker"]
market = market_all[market_all["ticker"] == ticker_principal].copy().sort_values("date").reset_index(drop=True)

print("Ticker principal:", ticker_principal)
print("N pontos:", len(market))
print("K mediano:", market["strike_price"].median())
print("Spot min/max:", market["spot_price"].min(), market["spot_price"].max())
print("Tau min/max:", market["tau"].min(), market["tau"].max())
print("Moneyness min/max:", market["moneyness"].min(), market["moneyness"].max())
print("IV mediana:", market["impl_volatility"].median())

market.to_csv(TAB_DIR / "00_market_selected_clean.csv", index=False)
display(market.head())

In [ ]:
# Figuras iniciais do mercado
fig, axes = plt.subplots(2, 2, figsize=(10.5, 6.5))

axes[0, 0].plot(market["date"], market["spot_price"], "-", color=COLORS["Black-Scholes"])
axes[0, 0].set_title("NDX spot")
axes[0, 0].set_xlabel("Date")
axes[0, 0].set_ylabel("Spot")

axes[0, 1].plot(market["date"], market["market_price"], "ko-", markersize=3)
axes[0, 1].set_title("Option market midpoint")
axes[0, 1].set_xlabel("Date")
axes[0, 1].set_ylabel("Option price")

axes[1, 0].hist(market["moneyness"], bins=16, color="#8FB131", edgecolor="black", linewidth=0.4)
axes[1, 0].set_title("Moneyness distribution")
axes[1, 0].set_xlabel(r"$S/K$")
axes[1, 0].set_ylabel("Count")

axes[1, 1].hist(market["tau"], bins=16, color="#E19C24", edgecolor="black", linewidth=0.4)
axes[1, 1].set_title("Time to maturity distribution")
axes[1, 1].set_xlabel(r"$\tau$")
axes[1, 1].set_ylabel("Count")

plt.tight_layout()
plt.savefig(OUT_DIR / "00_market_data_overview.png")
plt.show()

## 3. Carregar Greeks e selecionar melhores modelos

Carrega `experimentos_pinn/greeks/greeks_summary.csv`. Para gerar esse arquivo, rode antes:

```bash
python calculate_greeks_from_runs.py
```

Este notebook remove CQNN e trabalha só com **CLASSICO, QNN e HQNN**.

In [ ]:
from calculate_greeks_from_runs import load_model

GREEKS_SUMMARY = RESULTS_DIR / "greeks" / "greeks_summary.csv"
if not GREEKS_SUMMARY.exists():
    raise FileNotFoundError(
        f"Não encontrei {GREEKS_SUMMARY}. Rode primeiro: python calculate_greeks_from_runs.py"
    )

greeks_raw = pd.read_csv(GREEKS_SUMMARY)

if "family" not in greeks_raw.columns:
    greeks_raw["family"] = greeks_raw["model_type"].apply(normalize_family)
else:
    greeks_raw["family"] = greeks_raw["family"].apply(normalize_family)

# Sem CQNN
allowed_model_types = ["MLP", "ResNet", "QNN", "QPINN", "HQNN"]
greeks_raw = greeks_raw[greeks_raw["model_type"].isin(allowed_model_types)].copy()
greeks_raw = greeks_raw[greeks_raw["family"].isin(FAMILIES)].copy()

# Garantir RMSE se só houver MSE e converter métricas principais
for key in ["V", "delta", "gamma", "theta"]:
    rmse_col = f"RMSE_{key}_global"
    mse_col = f"MSE_{key}_global"
    mae_col = f"MAE_{key}_global"
    if rmse_col not in greeks_raw.columns and mse_col in greeks_raw.columns:
        greeks_raw[rmse_col] = np.sqrt(pd.to_numeric(greeks_raw[mse_col], errors="coerce"))
    for col in [rmse_col, mse_col, mae_col]:
        if col in greeks_raw.columns:
            greeks_raw[col] = pd.to_numeric(greeks_raw[col], errors="coerce")

if "num_params" in greeks_raw.columns:
    greeks_raw["num_params"] = pd.to_numeric(greeks_raw["num_params"], errors="coerce")
else:
    greeks_raw["num_params"] = np.nan

# ------------------------------------------------------------
# Aplicar seleção por hidden, blocks, qubits, layers, seed etc.
# ------------------------------------------------------------

def _same_type_filter(series, allowed_values):
    """Filtro robusto para valores numéricos ou texto."""
    if allowed_values is None:
        return pd.Series(True, index=series.index)
    if len(allowed_values) == 0:
        return pd.Series(False, index=series.index)

    s_num = pd.to_numeric(series, errors="coerce")
    vals_num = pd.to_numeric(pd.Series(allowed_values), errors="coerce")

    if vals_num.notna().all() and s_num.notna().any():
        return s_num.isin(vals_num.astype(float).tolist())

    return series.astype(str).isin([str(v) for v in allowed_values])


def apply_selection(greeks_df):
    selected_parts = []
    selection_log = []

    for fam in FAMILIES:
        fam_all = greeks_df[greeks_df["family"] == fam].copy()
        fam_sel = fam_all.copy()

        exact_runs = EXACT_RUNS_BY_FAMILY.get(fam, [])
        if exact_runs:
            fam_sel = fam_sel[fam_sel["run_id"].astype(str).isin([str(x) for x in exact_runs])].copy()
        else:
            rules = MODEL_SELECTION.get(fam, {})
            for col, allowed_values in rules.items():
                if allowed_values is None:
                    continue
                if col not in fam_sel.columns:
                    print(f"[aviso] coluna '{col}' não existe para filtro de {fam}; ignorando.")
                    continue
                fam_sel = fam_sel[_same_type_filter(fam_sel[col], allowed_values)].copy()

        max_params = MAX_PARAMS_BY_FAMILY.get(fam, None)
        if max_params is not None and "num_params" in fam_sel.columns:
            fam_sel = fam_sel[fam_sel["num_params"] <= float(max_params)].copy()

        if fam_sel.empty and ALLOW_FALLBACK_TO_UNFILTERED_FAMILY and not fam_all.empty:
            print(f"[aviso] filtros deixaram {fam} vazio; usando todas as runs de {fam}.")
            fam_sel = fam_all.copy()

        selection_log.append({
            "family": fam,
            "n_before": len(fam_all),
            "n_after": len(fam_sel),
            "exact_runs_used": bool(exact_runs),
            "max_params": max_params,
        })
        selected_parts.append(fam_sel)

    if len(selected_parts) == 0:
        return greeks_df.iloc[0:0].copy(), pd.DataFrame(selection_log)

    return pd.concat(selected_parts, ignore_index=True), pd.DataFrame(selection_log)


greeks, selection_log = apply_selection(greeks_raw)

print("Resumo da seleção:")
display(selection_log)

if SELECTION_SCORE_COL not in greeks.columns:
    raise ValueError(
        f"SELECTION_SCORE_COL='{SELECTION_SCORE_COL}' não existe. "
        f"Colunas disponíveis: {list(greeks.columns)}"
    )

greeks[SELECTION_SCORE_COL] = pd.to_numeric(greeks[SELECTION_SCORE_COL], errors="coerce")

# Melhor run por família dentro do subconjunto selecionado
best_rows = []
for fam, g in greeks.groupby("family"):
    g = g.dropna(subset=[SELECTION_SCORE_COL]).copy()
    if len(g):
        g = g.sort_values([SELECTION_SCORE_COL, "num_params"])
        best_rows.append(g.iloc[0])

best_models = pd.DataFrame(best_rows).reset_index(drop=True)

cols = [c for c in [
    "family", "model_type", "run_id", SELECTION_SCORE_COL, "RMSE_V_global", "MAE_V_global",
    "RMSE_delta_global", "RMSE_gamma_global", "RMSE_theta_global", "num_params",
    "hidden", "blocks", "n_qubits", "n_layers", "entangler", "seed"
] if c in best_models.columns]

print("Melhores modelos selecionados:")
display(best_models[cols])

# Salvar tabelas
selection_log.to_csv(TAB_DIR / "01a_selection_log.csv", index=False)
greeks_raw.to_csv(TAB_DIR / "01_greeks_summary_no_cqnn_all.csv", index=False)
greeks.to_csv(TAB_DIR / "01b_greeks_summary_no_cqnn_filtered.csv", index=False)
best_models.to_csv(TAB_DIR / "02_best_models_no_cqnn_selected.csv", index=False)


In [ ]:
# Carregar modelos selecionados
loaded_models = {}
selected_info = []

for _, row in best_models.iterrows():
    fam = row["family"]
    model_type = row["model_type"]
    run_id = row["run_id"]
    run_dir = RESULTS_DIR / "runs" / str(model_type) / str(run_id)

    model, cfg, state_path = load_model(run_dir, device=DEVICE)
    loaded_models[fam] = model

    selected_info.append({
        "family": fam,
        "model_type": model_type,
        "run_id": run_id,
        "num_params": row.get("num_params", np.nan),
        "state_path": str(state_path),
    })

selected_info = pd.DataFrame(selected_info)
display(selected_info)

## 4. Experimento 1 — Black–Scholes sintético 1D

Aqui a comparação é controlada: curvas `S → V(S,t)` no domínio sintético do treino. Esta é a parte mais próxima da validação PINN/Trahan: solução analítica contra modelo, com comparação por parâmetros.

In [ ]:
S_grid = np.linspace(0.1, TRAIN_S_MAX, 240)
t_cuts_price = [0.0, 0.5, 0.9, 0.99]
t_cuts_greeks = [0.0, 0.5, 0.9]

synthetic_rows = []

for t0 in t_cuts_price:
    t_arr = np.full_like(S_grid, t0, dtype=float)
    V_bs, delta_bs, gamma_bs, theta_bs = bs_call_greeks_pde_t(S_grid, t_arr)

    base = pd.DataFrame({
        "S": S_grid,
        "t": t_arr,
        "tau": TRAIN_T - t_arr,
        "V_BS": V_bs,
        "delta_BS": delta_bs,
        "gamma_BS": gamma_bs,
        "theta_BS": theta_bs,
    })

    for fam, model in loaded_models.items():
        Vp, dp, gp, tp = predict_price_greeks_synthetic(model, S_grid, t_arr)
        tmp = base.copy()
        tmp["family"] = fam
        tmp["V_pred"] = Vp
        tmp["delta_pred"] = dp
        tmp["gamma_pred"] = gp
        tmp["theta_pred"] = tp
        synthetic_rows.append(tmp)

synthetic_pred = pd.concat(synthetic_rows, ignore_index=True)
synthetic_pred["moneyness"] = synthetic_pred["S"] / TRAIN_K
synthetic_pred["moneyness_region"] = pd.cut(
    synthetic_pred["moneyness"],
    bins=[-np.inf, 0.97, 1.03, np.inf],
    labels=["OTM", "ATM", "ITM"],
)

synthetic_pred.to_csv(TAB_DIR / "03_synthetic_1d_predictions.csv", index=False)
display(synthetic_pred.head())

In [ ]:
# Curvas de preço sintético
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
axes = axes.ravel()

for ax, t0 in zip(axes, t_cuts_price):
    sub_bs = synthetic_pred[synthetic_pred["t"] == t0].drop_duplicates("S")
    ax.plot(sub_bs["S"], sub_bs["V_BS"], "k--", linewidth=2, label="Black-Scholes")

    for fam in FAMILIES:
        sub = synthetic_pred[(synthetic_pred["t"] == t0) & (synthetic_pred["family"] == fam)]
        if len(sub):
            ax.plot(sub["S"], sub["V_pred"], linewidth=1.8, color=COLORS[fam], label=fam)

    ax.axvline(TRAIN_K, color="0.4", linestyle=":", linewidth=1)
    ax.set_title(fr"$t={t0:.2f}$")
    ax.set_xlabel(r"$S$")
    ax.set_ylabel(r"$V(S,t)$")

axes[0].legend(ncol=2)
plt.suptitle("Synthetic Black-Scholes price cuts", y=1.01)
savefig("01_synthetic_price_cuts.png")

In [ ]:
# Erro absoluto do preço sintético
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
axes = axes.ravel()

for ax, t0 in zip(axes, t_cuts_price):
    for fam in FAMILIES:
        sub = synthetic_pred[(synthetic_pred["t"] == t0) & (synthetic_pred["family"] == fam)]
        if len(sub):
            ax.plot(sub["S"], np.abs(sub["V_pred"] - sub["V_BS"]), linewidth=1.8, color=COLORS[fam], label=fam)
    ax.axvline(TRAIN_K, color="0.4", linestyle=":", linewidth=1)
    ax.set_yscale("log")
    ax.set_title(fr"$t={t0:.2f}$")
    ax.set_xlabel(r"$S$")
    ax.set_ylabel(r"$|V_{pred}-V_{BS}|$")

axes[0].legend(ncol=2)
plt.suptitle("Synthetic Black-Scholes absolute error", y=1.01)
savefig("02_synthetic_price_abs_error.png")

In [ ]:
# Métricas sintéticas por família, tempo e região
metric_rows = []

for (fam, t0), g in synthetic_pred.groupby(["family", "t"]):
    for target in ["V", "delta", "gamma", "theta"]:
        m = metrics(g[f"{target}_BS"], g[f"{target}_pred"])
        m["family"] = fam
        m["t"] = t0
        m["region"] = "Global"
        m["target"] = target
        metric_rows.append(m)

    for region, gr in g.groupby("moneyness_region"):
        for target in ["V", "delta", "gamma", "theta"]:
            m = metrics(gr[f"{target}_BS"], gr[f"{target}_pred"])
            m["family"] = fam
            m["t"] = t0
            m["region"] = str(region)
            m["target"] = target
            metric_rows.append(m)

synthetic_metrics = pd.DataFrame(metric_rows)
synthetic_metrics.to_csv(TAB_DIR / "04_synthetic_metrics_by_region.csv", index=False)
display(synthetic_metrics.head(20))

## 4B. Teste sintético de 1 dólar

Aqui verificamos se o erro médio absoluto de preço no domínio sintético é menor que **1 dólar sintético**.  
Como o problema sintético usa `K=40`, `S_max=160`, `V_max=120`, o erro está em unidades desnormalizadas do preço da opção.


In [ ]:
# ============================================================
# TESTE SINTÉTICO — ERRO MÉDIO ABSOLUTO MENOR QUE 1 DÓLAR
# Usa as curvas sintéticas calculadas acima em synthetic_pred.
# ============================================================

# Erro ponto a ponto em dólar sintético
synthetic_pred["abs_error_V"] = np.abs(synthetic_pred["V_pred"] - synthetic_pred["V_BS"])
synthetic_pred["error_V"] = synthetic_pred["V_pred"] - synthetic_pred["V_BS"]

one_dollar_rows = []

# Global por família, usando todos os pontos S e todos os cortes t
for fam, g in synthetic_pred.groupby("family"):
    m = metrics(g["V_BS"], g["V_pred"])
    one_dollar_rows.append({
        "family": fam,
        "region": "Global_all_cuts",
        "N": int(m["N"]),
        "MAE_V": float(m["MAE"]),
        "RMSE_V": float(m["RMSE"]),
        "Bias_V": float(m["Bias"]),
        "Corr_V": float(m["Corr"]),
        "pass_mean_error_lt_1_dollar": bool(m["MAE"] < SYNTHETIC_ONE_DOLLAR_THRESHOLD),
    })

# Por corte de tempo
for (fam, t0), g in synthetic_pred.groupby(["family", "t"]):
    m = metrics(g["V_BS"], g["V_pred"])
    one_dollar_rows.append({
        "family": fam,
        "region": f"t={t0:.2f}",
        "N": int(m["N"]),
        "MAE_V": float(m["MAE"]),
        "RMSE_V": float(m["RMSE"]),
        "Bias_V": float(m["Bias"]),
        "Corr_V": float(m["Corr"]),
        "pass_mean_error_lt_1_dollar": bool(m["MAE"] < SYNTHETIC_ONE_DOLLAR_THRESHOLD),
    })

# Por moneyness, agregado em todos os cortes
for (fam, region), g in synthetic_pred.groupby(["family", "moneyness_region"]):
    m = metrics(g["V_BS"], g["V_pred"])
    one_dollar_rows.append({
        "family": fam,
        "region": str(region),
        "N": int(m["N"]),
        "MAE_V": float(m["MAE"]),
        "RMSE_V": float(m["RMSE"]),
        "Bias_V": float(m["Bias"]),
        "Corr_V": float(m["Corr"]),
        "pass_mean_error_lt_1_dollar": bool(m["MAE"] < SYNTHETIC_ONE_DOLLAR_THRESHOLD),
    })

synthetic_one_dollar = pd.DataFrame(one_dollar_rows)
synthetic_one_dollar.to_csv(TAB_DIR / "04b_synthetic_one_dollar_test.csv", index=False)

synthetic_one_dollar_global = synthetic_one_dollar[synthetic_one_dollar["region"] == "Global_all_cuts"].copy()

# Adicionar parâmetros/run selecionada
if "best_models" in globals() and len(best_models):
    keep = [c for c in ["family", "model_type", "run_id", "num_params", "hidden", "blocks", "n_qubits", "n_layers"] if c in best_models.columns]
    synthetic_one_dollar_global = synthetic_one_dollar_global.merge(best_models[keep], on="family", how="left")

print(f"Critério: MAE de preço sintético < {SYNTHETIC_ONE_DOLLAR_THRESHOLD} dólar")
display(synthetic_one_dollar_global.sort_values("MAE_V"))

display(synthetic_one_dollar.sort_values(["family", "region"]))

# Plot: MAE global por família, com linha de 1 dólar
plt.figure(figsize=(6.4, 4.1))
plot_df = synthetic_one_dollar_global.sort_values("MAE_V")
bar_colors = ["#4C9A2A" if ok else "#C44E52" for ok in plot_df["pass_mean_error_lt_1_dollar"]]
plt.bar(plot_df["family"], plot_df["MAE_V"], color=bar_colors)
plt.axhline(SYNTHETIC_ONE_DOLLAR_THRESHOLD, color="black", linestyle="--", linewidth=1.2, label="1 dólar")
plt.ylabel("MAE de preço sintético")
plt.xlabel("Família")
plt.title("Teste sintético: erro médio menor que 1 dólar")
plt.legend()
savefig("03b_synthetic_one_dollar_mae_by_family.png")

# Plot: MAE vs parâmetros, estilo Trahan, com linha de 1 dólar
if "num_params" in greeks.columns:
    plt.figure(figsize=(7.0, 4.4))

    # usa métrica sintética do greeks_summary se existir; senão usa RMSE como proxy visual
    y_col = "MAE_V_global" if "MAE_V_global" in greeks.columns else "RMSE_V_global"
    y_label = "MAE price" if y_col == "MAE_V_global" else "RMSE price"

    for fam in FAMILIES:
        g = greeks[(greeks["family"] == fam) & greeks[y_col].notna()].copy()
        if len(g):
            plt.scatter(
                g["num_params"], g[y_col],
                s=42, alpha=0.78, label=fam,
                color=COLORS[fam], edgecolor="black", linewidth=0.25,
            )

    plt.axhline(SYNTHETIC_ONE_DOLLAR_THRESHOLD, color="black", linestyle="--", linewidth=1.2, label="1 dólar")
    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel("Number of trainable parameters")
    plt.ylabel(y_label)
    plt.title("Synthetic error vs parameters with 1-dollar threshold")
    plt.legend()
    savefig("03c_synthetic_error_vs_params_one_dollar.png")


## 5. Experimento 2 — Greeks sintéticas 1D

Greeks contra Black–Scholes no domínio sintético. Isso é importante porque preço pode parecer correto mesmo quando derivadas são instáveis.

In [ ]:
for greek in ["delta", "gamma", "theta"]:
    fig, axes = plt.subplots(1, len(t_cuts_greeks), figsize=(14, 3.8), sharex=True)
    if len(t_cuts_greeks) == 1:
        axes = [axes]

    for ax, t0 in zip(axes, t_cuts_greeks):
        sub_bs = synthetic_pred[synthetic_pred["t"] == t0].drop_duplicates("S")
        ax.plot(sub_bs["S"], sub_bs[f"{greek}_BS"], "k--", linewidth=2, label="Black-Scholes")

        for fam in FAMILIES:
            sub = synthetic_pred[(synthetic_pred["t"] == t0) & (synthetic_pred["family"] == fam)]
            if len(sub):
                ax.plot(sub["S"], sub[f"{greek}_pred"], linewidth=1.7, color=COLORS[fam], label=fam)

        ax.axvline(TRAIN_K, color="0.4", linestyle=":", linewidth=1)
        ax.set_title(fr"$t={t0:.2f}$")
        ax.set_xlabel(r"$S$")
        ax.set_ylabel(greek)

    axes[0].legend(ncol=2)
    plt.suptitle(f"Synthetic {greek} cuts", y=1.04)
    savefig(f"03_synthetic_{greek}_cuts.png")

## 6. Experimento 3 — comparação estilo Trahan: erro vs parâmetros

Aqui não usamos tempo de cálculo. A pergunta é: **qual arquitetura entrega menor erro para um orçamento de parâmetros?**

In [ ]:
# Scatter erro vs parâmetros para V e Greeks
for metric_col, ylabel in [
    ("RMSE_V_global", "RMSE price"),
    ("RMSE_delta_global", "RMSE Delta"),
    ("RMSE_gamma_global", "RMSE Gamma"),
    ("RMSE_theta_global", "RMSE Theta"),
]:
    if metric_col not in greeks.columns:
        continue

    plt.figure(figsize=(6.8, 4.4))

    for fam in FAMILIES:
        g = greeks[(greeks["family"] == fam) & greeks[metric_col].notna()].copy()
        if len(g):
            plt.scatter(
                g["num_params"], g[metric_col],
                s=42, alpha=0.78, label=fam,
                color=COLORS[fam], edgecolor="black", linewidth=0.25,
            )

    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel("Number of trainable parameters")
    plt.ylabel(ylabel)
    plt.title(f"Trahan-style comparison: {ylabel} vs parameters")
    plt.legend()
    savefig(f"04_trahan_scatter_{metric_col}.png")

In [ ]:
# Melhor e mediana por família
summary_rows = []
for fam, g in greeks.groupby("family"):
    if fam not in FAMILIES:
        continue
    row = {"family": fam, "N": len(g)}
    for metric_col in ["RMSE_V_global", "RMSE_delta_global", "RMSE_gamma_global", "RMSE_theta_global"]:
        if metric_col in g.columns:
            row[f"best_{metric_col}"] = g[metric_col].min()
            row[f"median_{metric_col}"] = g[metric_col].median()
    row["median_params"] = g["num_params"].median()
    summary_rows.append(row)

family_summary = pd.DataFrame(summary_rows)
family_summary.to_csv(TAB_DIR / "05_family_summary_trahan_style.csv", index=False)
display(family_summary)

## 7. Experimento 4 — mercado real 1D com normalização por strike

Agora usamos o contrato real selecionado no início. O modelo foi treinado em escala sintética com `K=40`; portanto, antes de avaliar no mercado, reescalamos por strike:

\[
S_{scaled} = S_{market}\,rac{K_{train}}{K_{market}},
\qquad
V_{scaled} = V_{market}\,rac{K_{train}}{K_{market}}.
\]

In [ ]:
# Normalização por strike para o contrato real
market_pred = market.copy()

K_market = market_pred["strike_price"].median()
scale = TRAIN_K / K_market

market_pred["S_scaled"] = market_pred["spot_price"] * scale
market_pred["K_scaled"] = market_pred["strike_price"] * scale
market_pred["market_price_scaled"] = market_pred["market_price"] * scale

market_pred["tau_model"] = market_pred["tau"].clip(0, TRAIN_T)
market_pred["t_model"] = (TRAIN_T - market_pred["tau_model"]).clip(0, TRAIN_T)

market_pred["S_norm"] = market_pred["S_scaled"] / TRAIN_S_MAX
market_pred["t_norm"] = market_pred["t_model"] / TRAIN_T

print("scale:", scale)
print("S_norm min/max:", market_pred["S_norm"].min(), market_pred["S_norm"].max())
print("t_norm min/max:", market_pred["t_norm"].min(), market_pred["t_norm"].max())

# Benchmarks Black-Scholes: IV de mercado e sigma do treino
(
    market_pred["bs_iv_price"], market_pred["bs_iv_delta"], market_pred["bs_iv_gamma"], market_pred["bs_iv_theta"]
) = bs_call_greeks_tau(
    S=market_pred["spot_price"],
    tau=market_pred["tau_model"],
    K=market_pred["strike_price"],
    r=market_pred["r"],
    sigma=market_pred["impl_volatility"],
)

(
    market_pred["bs_train_price"], market_pred["bs_train_delta"], market_pred["bs_train_gamma"], market_pred["bs_train_theta"]
) = bs_call_greeks_tau(
    S=market_pred["spot_price"],
    tau=market_pred["tau_model"],
    K=market_pred["strike_price"],
    r=market_pred["r"],
    sigma=TRAIN_SIGMA,
)

for fam, model in loaded_models.items():
    Vp, dp, gp, tp = predict_price_greeks_market_scaled(model, market_pred, scale=scale)
    market_pred[f"{fam}_price"] = Vp
    market_pred[f"{fam}_delta"] = dp
    market_pred[f"{fam}_gamma"] = gp
    market_pred[f"{fam}_theta"] = tp

market_pred.to_csv(TAB_DIR / "06_market_predictions_normalized.csv", index=False)
display(market_pred.head())

In [ ]:
# Curva temporal mercado vs modelos
plt.figure(figsize=(9.5, 4.8))

plt.plot(market_pred["date"], market_pred["market_price"], "ko", markersize=4, label="Market")
plt.plot(market_pred["date"], market_pred["bs_iv_price"], color=COLORS["Black-Scholes IV"], linewidth=2, label="Black-Scholes IV")
plt.plot(market_pred["date"], market_pred["bs_train_price"], color=COLORS["Black-Scholes train σ"], linestyle="--", linewidth=2, label="Black-Scholes train σ")

for fam in FAMILIES:
    col = f"{fam}_price"
    if col in market_pred.columns:
        plt.plot(market_pred["date"], market_pred[col], linewidth=2, color=COLORS[fam], label=fam)

plt.xlabel("Date")
plt.ylabel("Option price")
plt.title("Real market NDX call: price curve")
plt.legend(ncol=2)
savefig("05_market_price_curve_time.png")

In [ ]:
# Curva preço contra spot
spot_df = market_pred.sort_values("spot_price")

plt.figure(figsize=(8.5, 4.8))
plt.plot(spot_df["spot_price"], spot_df["market_price"], "ko", markersize=4, label="Market")
plt.plot(spot_df["spot_price"], spot_df["bs_iv_price"], color=COLORS["Black-Scholes IV"], linewidth=2, label="Black-Scholes IV")
plt.plot(spot_df["spot_price"], spot_df["bs_train_price"], color=COLORS["Black-Scholes train σ"], linestyle="--", linewidth=2, label="Black-Scholes train σ")

for fam in FAMILIES:
    col = f"{fam}_price"
    if col in spot_df.columns:
        plt.plot(spot_df["spot_price"], spot_df[col], linewidth=2, color=COLORS[fam], label=fam)

plt.xlabel(r"Spot price $S$")
plt.ylabel("Option price")
plt.title("Real market NDX call: price vs spot")
plt.legend(ncol=2)
savefig("06_market_price_curve_spot.png")

In [ ]:
# Métricas globais contra mercado
price_methods = {
    "Black-Scholes IV": "bs_iv_price",
    "Black-Scholes train σ": "bs_train_price",
}
for fam in FAMILIES:
    col = f"{fam}_price"
    if col in market_pred.columns:
        price_methods[fam] = col

market_metric_rows = []
for name, col in price_methods.items():
    m = metrics(market_pred["market_price"], market_pred[col])
    m["method"] = name
    m["NegPredShare"] = float(np.mean(np.asarray(market_pred[col]) < 0))
    m["MinPred"] = float(np.nanmin(market_pred[col]))
    market_metric_rows.append(m)

market_metrics_global = pd.DataFrame(market_metric_rows).set_index("method").sort_values("RMSE")
market_metrics_global.to_csv(TAB_DIR / "07_market_metrics_global.csv")
display(market_metrics_global)

plt.figure(figsize=(7.2, 4.2))
plot_df = market_metrics_global.reset_index()
plt.bar(plot_df["method"], plot_df["RMSE"])
plt.xticks(rotation=35, ha="right")
plt.ylabel("RMSE vs market")
plt.title("Market RMSE by method")
savefig("07_market_rmse_by_method.png")

## 8. Experimento 5 — mercado por região financeira

Compara erro contra mercado por moneyness e maturidade. Isso conecta o resultado aos regimes financeiros difíceis.

In [ ]:
market_pred["moneyness_region"] = pd.cut(
    market_pred["moneyness"],
    bins=[-np.inf, 0.97, 1.03, np.inf],
    labels=["OTM", "ATM", "ITM"],
)

market_pred["maturity_region"] = pd.qcut(
    market_pred["tau_model"],
    q=3,
    labels=["Short", "Medium", "Long"],
    duplicates="drop",
)

region_rows = []
for region_col in ["moneyness_region", "maturity_region"]:
    for region, g in market_pred.groupby(region_col):
        for name, col in price_methods.items():
            m = metrics(g["market_price"], g[col])
            m["region_type"] = region_col
            m["region"] = str(region)
            m["method"] = name
            region_rows.append(m)

market_metrics_region = pd.DataFrame(region_rows)
market_metrics_region.to_csv(TAB_DIR / "08_market_metrics_by_region.csv", index=False)
display(market_metrics_region)

for region_type in ["moneyness_region", "maturity_region"]:
    tmp = market_metrics_region[market_metrics_region["region_type"] == region_type].copy()
    pivot = tmp.pivot(index="region", columns="method", values="RMSE")
    ax = pivot.plot(kind="bar", figsize=(9, 4.3))
    ax.set_ylabel("RMSE vs market")
    ax.set_xlabel(region_type)
    ax.set_title(f"Market RMSE by {region_type}")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"08_market_rmse_by_{region_type}.png")
    plt.show()

In [ ]:
# Região difícil: ATM + maturidade curta
short_cut = market_pred["tau_model"].quantile(1/3)
hard = market_pred[
    market_pred["moneyness"].between(0.97, 1.03) &
    (market_pred["tau_model"] <= short_cut)
].copy()

hard_rows = []
for name, col in price_methods.items():
    m = metrics(hard["market_price"], hard[col])
    m["method"] = name
    hard_rows.append(m)

hard_metrics = pd.DataFrame(hard_rows).set_index("method").sort_values("RMSE")
hard_metrics.to_csv(TAB_DIR / "09_market_hard_region_metrics.csv")
print("N hard region:", len(hard))
display(hard_metrics)

## 9. Experimento 6 — Greeks no mercado

Não há Greek observado no mercado. Aqui a referência é Black–Scholes com volatilidade implícita de mercado. A pergunta é se os modelos mantêm derivadas estáveis e coerentes em dados reais.

In [ ]:
# Curvas de Greeks no mercado contra BS-IV
for greek in ["delta", "gamma", "theta"]:
    plt.figure(figsize=(9.5, 4.5))

    plt.plot(
        market_pred["date"], market_pred[f"bs_iv_{greek}"],
        color=COLORS["Black-Scholes IV"], linewidth=2.2, label=f"BS IV {greek}"
    )
    plt.plot(
        market_pred["date"], market_pred[f"bs_train_{greek}"],
        color=COLORS["Black-Scholes train σ"], linestyle="--", linewidth=2, label=f"BS train σ {greek}"
    )

    for fam in FAMILIES:
        col = f"{fam}_{greek}"
        if col in market_pred.columns:
            plt.plot(market_pred["date"], market_pred[col], linewidth=1.8, color=COLORS[fam], label=f"{fam}")

    plt.xlabel("Date")
    plt.ylabel(greek)
    plt.title(f"Market {greek}: models vs Black-Scholes IV")
    plt.legend(ncol=2)
    savefig(f"09_market_{greek}_curve.png")

In [ ]:
# Métricas de Greeks contra BS-IV no mercado
market_greek_rows = []
for greek in ["delta", "gamma", "theta"]:
    true_col = f"bs_iv_{greek}"
    for fam in FAMILIES:
        pred_col = f"{fam}_{greek}"
        if pred_col in market_pred.columns:
            m = metrics(market_pred[true_col], market_pred[pred_col])
            m["family"] = fam
            m["greek"] = greek
            market_greek_rows.append(m)

market_greek_metrics = pd.DataFrame(market_greek_rows).sort_values(["greek", "RMSE"])
market_greek_metrics.to_csv(TAB_DIR / "10_market_greek_metrics_vs_bs_iv.csv", index=False)
display(market_greek_metrics)

for greek in ["delta", "gamma", "theta"]:
    tmp = market_greek_metrics[market_greek_metrics["greek"] == greek].copy()
    if tmp.empty:
        continue
    plt.figure(figsize=(5.8, 3.8))
    plt.bar(tmp["family"], tmp["RMSE"])
    plt.ylabel(f"RMSE {greek} vs BS-IV")
    plt.title(f"Market Greek RMSE: {greek}")
    savefig(f"10_market_greek_rmse_{greek}.png")

## 10. Experimento 7 — síntese para o artigo

Esta seção junta a leitura sintética e de mercado, sem usar tempo de cálculo.

In [ ]:
# Tabela final resumida: sintético + mercado + teste de 1 dólar
final_rows = []

for fam in FAMILIES:
    row = {"family": fam}

    # Métricas sintéticas por cortes 1D
    gsyn = synthetic_metrics[
        (synthetic_metrics["family"] == fam) &
        (synthetic_metrics["region"] == "Global") &
        (synthetic_metrics["target"] == "V")
    ]
    row["synthetic_RMSE_V_mean_over_cuts"] = gsyn["RMSE"].mean() if len(gsyn) else np.nan
    row["synthetic_RMSE_V_best_over_cuts"] = gsyn["RMSE"].min() if len(gsyn) else np.nan
    row["synthetic_MAE_V_mean_over_cuts"] = gsyn["MAE"].mean() if len(gsyn) else np.nan

    # Teste de 1 dólar usando todos os pontos sintéticos gerados no notebook
    if "synthetic_one_dollar_global" in globals():
        gone = synthetic_one_dollar_global[synthetic_one_dollar_global["family"] == fam]
        if len(gone):
            row["synthetic_MAE_V_all_cuts"] = gone.iloc[0].get("MAE_V", np.nan)
            row["synthetic_pass_1_dollar"] = gone.iloc[0].get("pass_mean_error_lt_1_dollar", np.nan)

    # Mercado real
    if fam in market_metrics_global.index:
        row["market_RMSE_price"] = market_metrics_global.loc[fam, "RMSE"]
        row["market_MAE_price"] = market_metrics_global.loc[fam, "MAE"]
        row["market_bias_price"] = market_metrics_global.loc[fam, "Bias"]
        row["negative_price_share_market"] = market_metrics_global.loc[fam, "NegPredShare"]

    # Info da run selecionada
    gparams = best_models[best_models["family"] == fam]
    if len(gparams):
        row["num_params_selected"] = gparams.iloc[0].get("num_params", np.nan)
        row["selected_model_type"] = gparams.iloc[0].get("model_type", "")
        row["selected_run_id"] = gparams.iloc[0].get("run_id", "")
        for col in ["hidden", "blocks", "n_qubits", "n_layers", "seed"]:
            if col in gparams.columns:
                row[f"selected_{col}"] = gparams.iloc[0].get(col, np.nan)

    final_rows.append(row)

final_summary = pd.DataFrame(final_rows)
final_summary.to_csv(TAB_DIR / "11_final_article_summary.csv", index=False)
display(final_summary)


In [ ]:
print("Figuras salvas em:", OUT_DIR.resolve())
print("Tabelas salvas em:", TAB_DIR.resolve())
print("Arquivos principais:")
for p in sorted(TAB_DIR.glob("*.csv")):
    print(" -", p.name)